# Kaggriculture submission 2: intensive market-aware rotation

This is an independent second submission. It preserves the required packaging
contract while using a strategy that differs materially from submission 1:

- all 25 plots in the starting quadrant are managed;
- five farm hands are hired each day;
- strawberries are included alongside the original crops;
- premium melon/strawberry slots rotate into staples when visible prices or
  opponent production indicate oversupply;
- stock-pressure selling protects the 100-item shed, followed by an early
  terminal liquidation.

The archive contains **`main.py` at its root**, exposes **`agent(obs)`**, and the
agent performs no network or filesystem access during an episode.

Official references: [competition overview](https://www.kaggle.com/competitions/kaggriculture/overview), [rules](https://www.kaggle.com/competitions/kaggriculture/rules).

## 1. Install the same environment version used for local validation

This installation is only for building/testing the notebook. The submitted `main.py` itself has no external dependencies.

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments==1.32.7"

## 2. Write the required sub2 entrypoint

This version is intentionally a separate strategy rather than a renamed copy.
It uses every initially unlocked plot and reacts to shared market prices and the
opponent's visible crop mix. The implementation remains deterministic and
self-contained so the Kaggle loader can execute it reliably.

In [ ]:
%%writefile main.py
"""Kaggriculture submission 2: an intensive, market-aware crop agent.

This version deliberately differs from submission 1.  It works the complete
starting quadrant, includes strawberries, and rotates price-sensitive premium
crops into resilient staples when the market or opponent signals oversupply.
It is self-contained and performs no network or filesystem access.
"""


PASS = ["PASS"]
DESIRED_HANDS = 5
MAX_MARKET_ORDERS = 10
FORCED_SELL_STOCK = 55
LIQUIDATION_DAY = 25
FINAL_FARM_DAY = 28


CROPS = {
    "WHEAT": {
        "seed_cost": 10,
        "base_price": 25,
        "first_yield_day": 2,
        "harvest_day": 4,
        "last_plant_day": 24,
        "ongoing": False,
    },
    "CARROT": {
        "seed_cost": 20,
        "base_price": 35,
        "first_yield_day": 2,
        "harvest_day": 3,
        "last_plant_day": 25,
        "ongoing": False,
    },
    "TOMATO": {
        "seed_cost": 50,
        "base_price": 60,
        "first_yield_day": 8,
        "harvest_day": 8,
        "last_plant_day": 20,
        "ongoing": True,
    },
    "STRAWBERRY": {
        "seed_cost": 100,
        "base_price": 120,
        "first_yield_day": 10,
        "harvest_day": 10,
        "last_plant_day": 18,
        "ongoing": True,
    },
    "MELON": {
        "seed_cost": 80,
        "base_price": 250,
        "first_yield_day": 10,
        "harvest_day": 10,
        "last_plant_day": 18,
        "ongoing": False,
    },
}


# Every cell in the unlocked 5x5 NW quadrant is used.  Each slot has a primary
# crop and a staple fallback.  Premium crops rotate into the fallback when their
# current price is weak or the opponent is already producing them at scale.
PLOT_SLOTS = (
    (0, 0, "MELON", "CARROT"),
    (1, 0, "STRAWBERRY", "WHEAT"),
    (2, 0, "MELON", "CARROT"),
    (3, 0, "CARROT", "WHEAT"),
    (4, 0, "MELON", "CARROT"),
    (0, 1, "WHEAT", "CARROT"),
    (1, 1, "MELON", "CARROT"),
    (2, 1, "STRAWBERRY", "WHEAT"),
    (3, 1, "MELON", "CARROT"),
    (4, 1, "TOMATO", "CARROT"),
    (0, 2, "MELON", "CARROT"),
    (1, 2, "CARROT", "WHEAT"),
    (2, 2, "MELON", "CARROT"),
    (3, 2, "STRAWBERRY", "WHEAT"),
    (4, 2, "MELON", "CARROT"),
    (0, 3, "STRAWBERRY", "WHEAT"),
    (1, 3, "MELON", "CARROT"),
    (2, 3, "WHEAT", "CARROT"),
    (3, 3, "MELON", "CARROT"),
    (4, 3, "TOMATO", "CARROT"),
    (0, 4, "MELON", "CARROT"),
    (1, 4, "CARROT", "WHEAT"),
    (2, 4, "STRAWBERRY", "WHEAT"),
    (3, 4, "WHEAT", "CARROT"),
    (4, 4, "MELON", "CARROT"),
)


# (normal batch size, minimum ordinary sale price).  Stock pressure and the
# terminal liquidation window override these thresholds.
SELL_RULES = {
    "WHEAT": (16, 18),
    "CARROT": (12, 24),
    "TOMATO": (6, 36),
    "STRAWBERRY": (4, 78),
    "MELON": (6, 160),
}


STRAWBERRY_SHOPS = {
    "BRUNCH_SPOT",
    "ICE_CREAM_SHOP",
    "SMOOTHIE_SHOP",
    "FARMERS_MARKET",
}


def _safe_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _step_toward(position, target):
    """Take one deterministic Manhattan step toward a target plot."""
    x, y = position
    tx, ty = target
    dx = tx - x
    dy = ty - y
    if abs(dx) >= abs(dy) and dx:
        return ["EAST" if dx > 0 else "WEST"]
    if dy:
        return ["SOUTH" if dy > 0 else "NORTH"]
    return PASS


def _crop_counts(farm):
    counts = {crop: 0 for crop in CROPS}
    for row in (farm.get("tiles", []) or []):
        for tile in row:
            if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                crop = tile.get("crop")
                if crop in counts:
                    counts[crop] += 1
    return counts


def _choose_crop(primary, fallback, day, market, town, opponent_counts):
    """Choose a slot's next crop from visible price and competition pressure."""
    prices = (market or {}).get("prices", {}) or {}
    shops = set((town or {}).get("unlocked_shops", []) or [])
    price = _safe_int(prices.get(primary), CROPS[primary]["base_price"])
    opponent_count = opponent_counts.get(primary, 0)

    candidate = primary
    if primary == "MELON":
        # Melon's glut curve is steep, so avoid a second large wave when the
        # shared price is already depressed or the opponent is melon-heavy.
        if price < 160 or opponent_count >= 8:
            candidate = fallback
    elif primary == "STRAWBERRY":
        # Early strawberries establish one premium cycle.  Later cycles require
        # either healthy price support or visible town demand.
        town_support = bool(shops.intersection(STRAWBERRY_SHOPS))
        if price < 82 or (day >= 9 and not town_support and opponent_count >= 5):
            candidate = fallback
    elif primary == "TOMATO" and price < 34:
        candidate = fallback

    if day <= CROPS[candidate]["last_plant_day"]:
        return candidate
    if day <= CROPS[fallback]["last_plant_day"]:
        return fallback
    if day <= CROPS["CARROT"]["last_plant_day"]:
        return "CARROT"
    if day <= CROPS["WHEAT"]["last_plant_day"]:
        return "WHEAT"
    return None


def _tile_task(tile, crop_to_plant, day, hour):
    """Return (priority, action) for one managed plot, or None."""
    if tile is None:
        if crop_to_plant is not None and hour <= 20:
            return 3, ["PLANT", crop_to_plant]
        return None

    if tile == "LOCKED" or not isinstance(tile, dict):
        return None

    kind = tile.get("kind")
    if kind == "WEED":
        return 2, ["DIG"]
    if kind != "PLANT":
        return None

    crop = tile.get("crop")
    data = CROPS.get(crop)
    if data is None:
        return 2, ["DIG"]

    age = day - _safe_int(tile.get("planted_day"), day)
    yield_units = _safe_int(tile.get("yield_units"), 0)
    watered = bool(tile.get("watered_today", False))

    # Day 28 is the final useful harvest day: inventories reach the shed
    # overnight and can still be sold on day 29.
    if day >= FINAL_FARM_DAY and yield_units > 0 and age >= data["first_yield_day"]:
        return 0, ["HARVEST"]

    if data["ongoing"]:
        if yield_units > 0 and age >= data["first_yield_day"]:
            return 0, ["HARVEST"]
        if not watered:
            return 1, ["WATER"]
        return None

    if yield_units > 0 and age >= data["harvest_day"]:
        # Include the final daily bonus at peak age, then harvest next turn.
        if age == data["harvest_day"] and not watered:
            return 0, ["WATER"]
        return 0, ["HARVEST"]
    if not watered:
        return 1, ["WATER"]
    return None


def _planned_empty_crops(farm, day, market, town, opponent_counts):
    """Count seeds required by empty or weeded managed plots."""
    wanted = {crop: 0 for crop in CROPS}
    tiles = farm.get("tiles", []) or []
    for x, y, primary, fallback in PLOT_SLOTS:
        try:
            tile = tiles[y][x]
        except (IndexError, TypeError):
            continue
        if tile is None or (isinstance(tile, dict) and tile.get("kind") == "WEED"):
            crop = _choose_crop(
                primary, fallback, day, market, town, opponent_counts
            )
            if crop is not None:
                wanted[crop] += 1
    return wanted


def _build_tasks(farm, private, day, hour, market, town, opponent_counts):
    """Build plot tasks while preventing atomic over-request of any seed."""
    seeds_available = {
        crop: _safe_int((private.get("seeds", {}) or {}).get(crop), 0)
        for crop in CROPS
    }
    tiles = farm.get("tiles", []) or []
    tasks = []

    for x, y, primary, fallback in PLOT_SLOTS:
        try:
            tile = tiles[y][x]
        except (IndexError, TypeError):
            continue
        crop_to_plant = _choose_crop(
            primary, fallback, day, market, town, opponent_counts
        )
        task = _tile_task(tile, crop_to_plant, day, hour)
        if task is None:
            continue
        priority, action = task
        if action[0] == "PLANT":
            crop = action[1]
            if seeds_available[crop] <= 0:
                continue
            seeds_available[crop] -= 1
        tasks.append(
            {
                "priority": priority,
                "target": (x, y),
                "action": action,
            }
        )
    return tasks


def _assign_unit_actions(
    farm, private, day, hour, market, town, opponent_counts
):
    positions = [tuple(farm.get("farmer", (4, 4)))]
    positions.extend(tuple(pos) for pos in (farm.get("hands", []) or []))
    actions = [PASS for _ in positions]
    tasks = _build_tasks(
        farm, private, day, hour, market, town, opponent_counts
    )
    remaining_units = set(range(len(positions)))

    # Urgency dominates distance.  This ensures all harvests and daily watering
    # are handled before replacement planting.
    while tasks and remaining_units:
        choices = []
        for unit_index in remaining_units:
            ux, uy = positions[unit_index]
            for task_index, task in enumerate(tasks):
                tx, ty = task["target"]
                distance = abs(tx - ux) + abs(ty - uy)
                choices.append(
                    (
                        task["priority"],
                        distance,
                        task["target"][1],
                        task["target"][0],
                        unit_index,
                        task_index,
                    )
                )
        _, _, _, _, unit_index, task_index = min(choices)
        task = tasks.pop(task_index)
        remaining_units.remove(unit_index)
        if positions[unit_index] == task["target"]:
            actions[unit_index] = task["action"]
        else:
            actions[unit_index] = _step_toward(
                positions[unit_index], task["target"]
            )
    return actions


def _sell_orders(private, market, day):
    shed = private.get("shed", {}) or {}
    prices = (market or {}).get("prices", {}) or {}
    shed_total = sum(max(0, _safe_int(value)) for value in shed.values())
    stock_pressure = shed_total >= FORCED_SELL_STOCK
    terminal = day >= LIQUIDATION_DAY
    orders = []

    for item in ("MELON", "STRAWBERRY", "TOMATO", "CARROT", "WHEAT"):
        held = _safe_int(shed.get(item), 0)
        if held <= 0:
            continue
        batch, threshold = SELL_RULES[item]
        price = _safe_int(prices.get(item), CROPS[item]["base_price"])
        if terminal or stock_pressure or price >= threshold:
            quantity = held if terminal else min(held, batch)
            orders.append(["SELL", item, quantity])
    return orders


def _seed_orders(
    farm, private, day, market, town, opponent_counts, market_slots
):
    if market_slots <= 0 or day >= FINAL_FARM_DAY:
        return []

    wanted = _planned_empty_crops(
        farm, day, market, town, opponent_counts
    )
    seeds = private.get("seeds", {}) or {}
    cash = float(farm.get("money", 0))
    reserve = 250.0
    orders = []

    # Staples receive first claim on constrained cash; premium seeds follow.
    for crop in ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"):
        if len(orders) >= market_slots:
            break
        missing = wanted[crop] - _safe_int(seeds.get(crop), 0)
        if missing <= 0:
            continue
        cost = CROPS[crop]["seed_cost"]
        affordable = max(0, int((cash - reserve) // cost))
        quantity = min(missing, affordable)
        if quantity <= 0:
            continue
        orders.append(["BUY_SEED", crop, quantity])
        cash -= quantity * cost
    return orders


def _market_orders(
    farm, private, market, town, day, hour, opponent_counts
):
    orders = _sell_orders(private, market, day)

    # Daily hand prices are Fibonacci-scaled; five hands cost only 12 total and
    # provide enough capacity for the intensive 25-plot plan.
    if hour == 0 and day <= FINAL_FARM_DAY:
        current_hands = len(farm.get("hands", []) or [])
        for _ in range(max(0, DESIRED_HANDS - current_hands)):
            if len(orders) >= MAX_MARKET_ORDERS:
                break
            orders.append(["HIRE"])

    free_slots = MAX_MARKET_ORDERS - len(orders)
    orders.extend(
        _seed_orders(
            farm,
            private,
            day,
            market,
            town,
            opponent_counts,
            free_slots,
        )
    )
    return orders[:MAX_MARKET_ORDERS]


def agent(obs):
    """Required Kaggle entrypoint."""
    try:
        farms = obs.get("farms", []) or []
        player = _safe_int(obs.get("player"), 0)
        private = obs.get("private", {}) or {}
        if player < 0 or player >= len(farms):
            return {"farmer": PASS, "hands": [], "market": []}

        farm = farms[player]
        opponent = farms[1 - player] if len(farms) == 2 else {}
        opponent_counts = _crop_counts(opponent)
        day = _safe_int(obs.get("day"), 0)
        hour = _safe_int(obs.get("hour"), 0)
        market = obs.get("market", {}) or {}
        town = obs.get("town", {}) or {}

        market_orders = _market_orders(
            farm,
            private,
            market,
            town,
            day,
            hour,
            opponent_counts,
        )

        if day > FINAL_FARM_DAY:
            farmer_action = PASS
            hand_actions = [PASS for _ in (farm.get("hands", []) or [])]
        else:
            unit_actions = _assign_unit_actions(
                farm,
                private,
                day,
                hour,
                market,
                town,
                opponent_counts,
            )
            farmer_action = unit_actions[0] if unit_actions else PASS
            hand_actions = unit_actions[1:]

        return {
            "farmer": farmer_action,
            "hands": hand_actions,
            "market": market_orders,
        }
    except Exception:
        # Preserve a valid action if a future environment adds unexpected data.
        hands = []
        try:
            farms = obs.get("farms", []) or []
            player = _safe_int(obs.get("player"), 0)
            if 0 <= player < len(farms):
                hands = [PASS for _ in (farms[player].get("hands", []) or [])]
        except Exception:
            hands = []
        return {"farmer": PASS, "hands": hands, "market": []}

## 3. Check the entrypoint and action contract

This catches the filename/function mismatch that the tutorial's in-memory callable test misses.

In [ ]:
import ast
import importlib.util
import json
from pathlib import Path

main_path = Path("main.py")
assert main_path.is_file(), "main.py was not created"

tree = ast.parse(main_path.read_text(encoding="utf-8"), filename="main.py")
function_names = {node.name for node in tree.body if isinstance(node, ast.FunctionDef)}
assert "agent" in function_names, "main.py must expose def agent(obs)"

spec = importlib.util.spec_from_file_location("submission_agent", main_path)
submission_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission_agent)
assert callable(submission_agent.agent)

tiles = [
    [None if x < 5 and y < 5 else "LOCKED" for x in range(10)]
    for y in range(10)
]
dummy_obs = {
    "player": 0,
    "day": 0,
    "hour": 0,
    "farms": [{
        "money": 3000,
        "tiles": tiles,
        "farmer": [4, 4],
        "hands": [],
        "unlocked_quadrants": ["NW"],
        "hires_today": 0,
    }],
    "private": {"shed": {}, "seeds": {}, "inventories": [{}]},
    "market": {"inventory": {}, "prices": {}},
    "town": {"unlocked_shops": []},
}

action = submission_agent.agent(dummy_obs)
assert set(action) == {"farmer", "hands", "market"}
assert isinstance(action["farmer"], list) and action["farmer"]
assert isinstance(action["hands"], list)
assert isinstance(action["market"], list) and len(action["market"]) <= 10
json.dumps(action)
print("Entrypoint and JSON action contract: OK")
print(action)

## 4. Run full file-loader validation games

These 720-turn matches exercise the exact `main.py` path Kaggle will load. The
self-play match mirrors the Validation Episode, and the two built-in opponents
provide additional smoke tests. These rewards are validation diagnostics, not a
leaderboard prediction.

In [ ]:
from kaggle_environments import make

opponents = ["main.py", "starter", "random"]
for index, opponent in enumerate(opponents):
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": 20260831 + index},
        debug=True,
    )
    env.run(["main.py", opponent])
    final = env.steps[-1]
    statuses = [state.status for state in final]
    rewards = [state.reward for state in final]
    assert statuses == ["DONE", "DONE"], (opponent, statuses)
    print(f"vs {opponent:8s}: statuses={statuses}, rewards={rewards}")

print("Sub2 full 720-turn file-loader validation: OK")

## 5. Build and verify the submission archive

The member name must be exactly `main.py`, with no enclosing directory.

In [ ]:
import hashlib
import tarfile
from pathlib import Path

archive_path = Path("submission.tar.gz")
with tarfile.open(archive_path, "w:gz") as archive:
    archive.add("main.py", arcname="main.py", recursive=False)

with tarfile.open(archive_path, "r:gz") as archive:
    members = archive.getnames()
    assert members == ["main.py"], members
    archived_source = archive.extractfile("main.py").read()

assert archived_source == Path("main.py").read_bytes()
size_mib = archive_path.stat().st_size / (1024 * 1024)
assert size_mib < 100, f"Archive is too large: {size_mib:.2f} MiB"
sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()

print(f"Created: {archive_path.resolve()}")
print(f"Members: {members}")
print(f"Size: {size_mib:.4f} MiB")
print(f"SHA-256: {sha256}")

## 6. Submit this as the second entry

1. Attach the **Kaggriculture** competition and accept its rules.
2. Run every cell and save a successful notebook version.
3. Confirm the validation ends with `Sub2 full 720-turn file-loader validation: OK`.
4. Click **Submit to competition** and select `submission.tar.gz` from this
   notebook version's outputs.

Keep submission 1 active while this version is evaluated; Kaggle permits the
latest two submissions to remain active.